In [4]:
import os 
from dotenv import load_dotenv
from llama_index.core import SimpleDirectoryReader
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
from llama_index.core import Settings
from llama_index.vector_stores.pinecone import PineconeVectorStore
from pinecone import Pinecone
from llama_index.core import VectorStoreIndex
from llama_index.core.callbacks import CallbackManager,LlamaDebugHandler

load_dotenv()

INDEX_NAME="llamaindex-practice"
EMBEDDING_DIMENSION=1536
Settings.chunk_size=512
Settings.chunk_overlap=50

#connect to pinecone
print("Connect to Pinecone")
pc=Pinecone(
    api_key=os.getenv("PINECONE_API_KEY"),
)
#callbacks
debug_handler=LlamaDebugHandler(print_trace_on_end=True)
#create callback manager
callback_manager=CallbackManager(handlers=[debug_handler])
#attach to settings

Settings.callback_manager=callback_manager

pinecone_index=pc.Index(INDEX_NAME)
vector_store=PineconeVectorStore(pinecone_index=pinecone_index)

stats=pinecone_index.describe_index_stats()
print(stats.total_vector_count)

#loading documents
documents=SimpleDirectoryReader(
    input_dir="llamaindex-docs",
    required_exts=[".md"],
    num_files_limit=5          
)
documents=documents.load_data()

pipeline=IngestionPipeline(
    transformations=[
        SentenceSplitter(chunk_size=Settings.chunk_size,chunk_overlap=Settings.chunk_overlap),
        OpenAIEmbedding(model="text-embedding-3-small")],
        vector_store=vector_store
    
)

pipeline_nodes=pipeline.run(documents=documents, show_progress=True, num_workers=4)
if pipeline_nodes:
    if pipeline_nodes[0].embedding:
        print(len(pipeline_nodes[0].embedding))
index=VectorStoreIndex.from_vector_store(vector_store=vector_store)
query_engine=index.as_query_engine()

response=query_engine.query("What is LlamaIndex")
print(f"Query Response: {response}")





Connect to Pinecone


2026-02-16 17:51:31,673 - WARNING - Removing unpickleable private attribute _chunking_tokenizer_fn
2026-02-16 17:51:31,675 - WARNING - Removing unpickleable private attribute _split_fns
2026-02-16 17:51:31,676 - WARNING - Removing unpickleable private attribute _sub_sentence_split_fns
2026-02-16 17:51:31,676 - WARNING - Removing unpickleable private attribute _chunking_tokenizer_fn
2026-02-16 17:51:31,677 - WARNING - Removing unpickleable private attribute _split_fns
2026-02-16 17:51:31,677 - WARNING - Removing unpickleable private attribute _sub_sentence_split_fns
2026-02-16 17:51:31,718 - WARNING - Removing unpickleable private attribute _chunking_tokenizer_fn
2026-02-16 17:51:31,719 - WARNING - Removing unpickleable private attribute _split_fns
2026-02-16 17:51:31,719 - WARNING - Removing unpickleable private attribute _sub_sentence_split_fns
2026-02-16 17:51:31,720 - WARNING - Removing unpickleable private attribute _chunking_tokenizer_fn
2026-02-16 17:51:31,720 - WARNING - Removin

11


Upserted vectors: 100%|██████████| 11/11 [00:00<00:00, 26.13it/s]

1536
**********
Trace: index_construction
**********



2026-02-16 17:51:36,461 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-02-16 17:51:37,565 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


**********
Trace: query
    |_CBEventType.QUERY -> 1.629656 seconds
      |_CBEventType.RETRIEVE -> 0.714736 seconds
        |_CBEventType.EMBEDDING -> 0.522415 seconds
      |_CBEventType.SYNTHESIZE -> 0.91492 seconds
        |_CBEventType.TEMPLATING -> 0.0 seconds
        |_CBEventType.LLM -> 0.912918 seconds
**********
Query Response: LlamaIndex is a service for document processing and search that powers LlamaCloud.


In [3]:
pinecone_index=pc.Index(INDEX_NAME)
vector_store=PineconeVectorStore(pinecone_index=pinecone_index)
index=VectorStoreIndex.from_vector_store(vector_store=vector_store)
query="How do i use llamaindex with pinecone?"
query_engine=index.as_query_engine()
response=query_engine.query(query)
print(response)

2026-02-16 17:23:37,069 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-02-16 17:23:39,490 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


You can use LlamaIndex with Pinecone by integrating LlamaCloud's document processing and search capabilities with Pinecone's vector database integration. This allows you to transform document collections into searchable knowledge bases using LlamaIndex's features and leverage Pinecone's vector database for efficient storage and retrieval of data.


In [2]:
!python -m pip -q install pinecone llama-index-vector-stores-pinecone

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
